# FitGuide AI Fitness & Nutrition Coach Agent

This notebook builds the FitGuide AI agent for ITAI 2376 Final Course Project AI Agent. FitGuide is a single-agent system that uses LangChain, Gemini, tool calling, a profile classifier, and session-based memory to generate personalized fitness and nutrition recommendations.

The notebook is organized step by step so the full agent pipeline is easy to follow: setup, data, helper functions, tools, agent creation, memory, and demo scenarios.

# Step 1 — Install Required Packages
This cell installs the Python libraries needed for the FitGuide AI agent. LangChain is used as the agent framework, `langchain-google-genai` connects the agent to Google Gemini, and pandas is used to organize the exercise dataset. We pin pandas to a Colab-friendly version to avoid package conflicts.

In [4]:
!pip install -q -U langchain langchain-google-genai
!pip install -q pandas==2.2.2

# Step 2 — Verify Pandas Version
This cell checks that pandas installed correctly. Pandas is used later to store and filter the exercise dataset.

In [5]:
import pandas as pd
print(pd.__version__)

2.2.2


# Step 3 — Load Gemini API Key Securely
This cell loads the Gemini API key from Google Colab Secrets. The key is not typed directly into the notebook, which keeps it safer and prevents it from being accidentally uploaded to GitHub.

In [6]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")

print("API key loaded:", os.environ["GOOGLE_API_KEY"] is not None)

API key loaded: True


# Step 4 — Create Exercise Dataset
This cell builds a small custom exercise dataset using pandas. The dataset includes exercise name, muscle group, equipment, difficulty, goal type, sets, and reps. The agent uses this data through an exercise lookup tool.

In [7]:
import pandas as pd

exercise_data = [
    ["Push-Up", "Chest/Triceps", "Bodyweight", "Beginner", "general fitness", "3", "10-15"],
    ["Bodyweight Squat", "Legs", "Bodyweight", "Beginner", "fat loss", "3", "12-15"],
    ["Walking Lunges", "Legs", "Bodyweight", "Beginner", "fat loss", "3", "10 each leg"],
    ["Plank", "Core", "Bodyweight", "Beginner", "general fitness", "3", "30-45 sec"],
    ["Jumping Jacks", "Full Body", "Bodyweight", "Beginner", "fat loss", "3", "30 sec"],
    ["Glute Bridge", "Glutes", "Bodyweight", "Beginner", "general fitness", "3", "12-15"],

    ["Goblet Squat", "Legs", "Dumbbell", "Beginner", "fat loss", "3", "10-12"],
    ["Dumbbell Lunges", "Legs", "Dumbbell", "Beginner", "fat loss", "3", "10 each leg"],
    ["Dumbbell Romanian Deadlift", "Hamstrings/Glutes", "Dumbbell", "Beginner", "fat loss", "3", "10-12"],
    ["Dumbbell Step-Up", "Legs/Glutes", "Dumbbell", "Beginner", "fat loss", "3", "10 each leg"],

    ["Dumbbell Row", "Back", "Dumbbell", "Beginner", "muscle gain", "3", "8-12"],
    ["Dumbbell Shoulder Press", "Shoulders", "Dumbbell", "Beginner", "muscle gain", "3", "8-12"],
    ["Dumbbell Chest Press", "Chest", "Dumbbell", "Beginner", "muscle gain", "3", "8-12"],
    ["Dumbbell Bicep Curl", "Arms", "Dumbbell", "Beginner", "muscle gain", "3", "10-12"],
    ["Dumbbell Triceps Extension", "Arms", "Dumbbell", "Beginner", "muscle gain", "3", "10-12"],

    ["Dumbbell Farmer Carry", "Full Body/Core", "Dumbbell", "Beginner", "general fitness", "3", "30 sec"],
    ["Dumbbell Deadlift", "Back/Legs", "Dumbbell", "Beginner", "general fitness", "3", "10-12"],
    ["Standing Calf Raise", "Calves", "Bodyweight", "Beginner", "general fitness", "3", "12-20"],
]

exercise_df = pd.DataFrame(
    exercise_data,
    columns=[
        "exercise_name",
        "muscle_group",
        "equipment",
        "difficulty",
        "goal_type",
        "sets",
        "reps"
    ]
)

exercise_df

,exercise_name,muscle_group,equipment,difficulty,goal_type,sets,reps
0,Push-Up,Chest/Triceps,Bodyweight,Beginner,general fitness,3,10-15
1,Bodyweight Squat,Legs,Bodyweight,Beginner,fat loss,3,12-15
2,Walking Lunges,Legs,Bodyweight,Beginner,fat loss,3,10 each leg
3,Plank,Core,Bodyweight,Beginner,general fitness,3,30-45 sec
4,Jumping Jacks,Full Body,Bodyweight,Beginner,fat loss,3,30 sec
5,Glute Bridge,Glutes,Bodyweight,Beginner,general fitness,3,12-15
6,Goblet Squat,Legs,Dumbbell,Beginner,fat loss,3,10-12
7,Dumbbell Lunges,Legs,Dumbbell,Beginner,fat loss,3,10 each leg
8,Dumbbell Romanian Deadlift,Hamstrings/Glutes,Dumbbell,Beginner,fat loss,3,10-12
9,Dumbbell Step-Up,Legs/Glutes,Dumbbell,Beginner,fat loss,3,10 each leg


# Step 5 — Create Nutrition Data and Helper Functions
This cell creates the small demo nutrition database and the main helper functions. These functions classify the user profile, search for exercises, look up nutrition information, and estimate calorie/protein targets.

In [8]:
food_db = {
    "chicken breast": {"calories": 165, "protein": 31, "carbs": 0, "fat": 3.6},
    "rice": {"calories": 130, "protein": 2.7, "carbs": 28, "fat": 0.3},
    "egg": {"calories": 78, "protein": 6, "carbs": 0.6, "fat": 5},
    "oats": {"calories": 150, "protein": 5, "carbs": 27, "fat": 3},
    "salmon": {"calories": 208, "protein": 20, "carbs": 0, "fat": 13},
    "banana": {"calories": 105, "protein": 1.3, "carbs": 27, "fat": 0.4},
    "greek yogurt": {"calories": 100, "protein": 10, "carbs": 4, "fat": 0.7},
}

def classify_user_profile(goal, equipment, difficulty="Beginner"):
    goal_lower = goal.lower()
    equipment_lower = equipment.lower()

    if "fat" in goal_lower or "lose" in goal_lower or "weight loss" in goal_lower:
        plan_type = "fat loss"
    elif "muscle" in goal_lower or "bulk" in goal_lower or "gain" in goal_lower:
        plan_type = "muscle gain"
    else:
        plan_type = "general fitness"

    if "dumbbell" in equipment_lower:
        equipment_class = "Dumbbell"
    elif "bodyweight" in equipment_lower or "no equipment" in equipment_lower:
        equipment_class = "Bodyweight"
    else:
        equipment_class = "Bodyweight"

    return {
        "plan_type": plan_type,
        "equipment_class": equipment_class,
        "difficulty": difficulty
    }

def find_exercises(goal_type, equipment, difficulty="Beginner"):
    equipment = equipment.lower()

    if "dumbbell" in equipment:
        equipment = "Dumbbell"
    elif "bodyweight" in equipment:
        equipment = "Bodyweight"

    results = exercise_df[
        (exercise_df["goal_type"].str.contains(goal_type, case=False, na=False)) &
        (exercise_df["equipment"].str.contains(equipment, case=False, na=False)) &
        (exercise_df["difficulty"].str.contains(difficulty, case=False, na=False))
    ]

    if results.empty:
        results = exercise_df[
            (exercise_df["equipment"].str.contains(equipment, case=False, na=False))
        ]

    return results.to_dict(orient="records")

def get_food_info(food_name):
    food_name = food_name.lower().strip()
    return food_db.get(food_name, {"error": "Food not found in small demo database."})

def estimate_macros(weight_lbs, goal):
    goal = goal.lower()
    protein = round(weight_lbs * 0.8)

    if "fat" in goal or "lose" in goal:
        calories = round(weight_lbs * 12)
    elif "muscle" in goal or "gain" in goal:
        calories = round(weight_lbs * 16)
    else:
        calories = round(weight_lbs * 14)

    return {
        "estimated_daily_calories": calories,
        "estimated_daily_protein_grams": protein
    }

print("Nutrition data and helper functions loaded successfully.")

Nutrition data and helper functions loaded successfully.


# Step 6 — Test Helper Functions
Before connecting the functions to the agent, this cell tests them directly. This confirms that the profile classifier, exercise lookup, macro estimator, and food lookup are working correctly.

In [9]:
test_profile = classify_user_profile(
    goal="I want to lose fat",
    equipment="dumbbells",
    difficulty="Beginner"
)

print("Classified Profile:")
print(test_profile)

print("\nExercise Results:")
print(find_exercises(test_profile["plan_type"], test_profile["equipment_class"]))

print("\nMacro Estimate:")
print(estimate_macros(180, test_profile["plan_type"]))

print("\nFood Lookup:")
print(get_food_info("chicken breast"))

Classified Profile:
{'plan_type': 'fat loss', 'equipment_class': 'Dumbbell', 'difficulty': 'Beginner'}

Exercise Results:
[{'exercise_name': 'Goblet Squat', 'muscle_group': 'Legs', 'equipment': 'Dumbbell', 'difficulty': 'Beginner', 'goal_type': 'fat loss', 'sets': '3', 'reps': '10-12'}, {'exercise_name': 'Dumbbell Lunges', 'muscle_group': 'Legs', 'equipment': 'Dumbbell', 'difficulty': 'Beginner', 'goal_type': 'fat loss', 'sets': '3', 'reps': '10 each leg'}, {'exercise_name': 'Dumbbell Romanian Deadlift', 'muscle_group': 'Hamstrings/Glutes', 'equipment': 'Dumbbell', 'difficulty': 'Beginner', 'goal_type': 'fat loss', 'sets': '3', 'reps': '10-12'}, {'exercise_name': 'Dumbbell Step-Up', 'muscle_group': 'Legs/Glutes', 'equipment': 'Dumbbell', 'difficulty': 'Beginner', 'goal_type': 'fat loss', 'sets': '3', 'reps': '10 each leg'}]

Macro Estimate:
{'estimated_daily_calories': 2160, 'estimated_daily_protein_grams': 144}

Food Lookup:
{'calories': 165, 'protein': 31, 'carbs': 0, 'fat': 3.6}


# Step 7 — Convert Helper Functions into LangChain Tools
This cell wraps the helper functions as LangChain tools. The agent can call these tools during execution instead of relying only on generated text.

In [10]:
from langchain.tools import tool

@tool
def lookup_exercises(goal_type: str, equipment: str, difficulty: str = "Beginner") -> str:
    """
    Finds exercises that match the user's fitness goal, available equipment, and difficulty level.
    """
    results = find_exercises(goal_type, equipment, difficulty)
    return str(results)


@tool
def nutrition_lookup(food_name: str) -> str:
    """
    Looks up basic nutrition information for a food item.
    """
    result = get_food_info(food_name)
    return str(result)


@tool
def macro_estimator(weight_lbs: float, goal: str) -> str:
    """
    Estimates daily calories and protein grams based on user weight and fitness goal.
    """
    result = estimate_macros(weight_lbs, goal)
    return str(result)


tools = [lookup_exercises, nutrition_lookup, macro_estimator]

print("LangChain tools created successfully.")

LangChain tools created successfully.


# Step 8 — Create Initial FitGuide Agent
This cell creates the first version of the FitGuide agent using LangChain and Gemini. The system prompt explains the agent's role and requires a structured fitness and nutrition response.

In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.4
)

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="""
You are FitGuide, an AI fitness and nutrition coach agent.

Your job:
1. Understand the user's fitness goal.
2. Use tools when needed.
3. Create a simple beginner-friendly plan.
4. Keep advice safe and general.
5. Do not claim to be a doctor, dietitian, or certified trainer.

Always respond using this format:

1. Goal Summary
2. Workout Recommendation
3. Nutrition Guidance
4. Estimated Calories and Protein
5. Weekly Coaching Tip
"""
)

print("FitGuide agent recreated successfully with Gemini 2.5 Flash.")

FitGuide agent recreated successfully with Gemini 2.5 Flash.


# Step 9 — Test Full Agent Pipeline
This cell tests the first full pipeline: user input is classified, passed to the agent, and the agent can call tools to create a personalized plan. This shows the flow from profile classification to final output.

In [12]:
user_goal = "I want to lose fat"
user_equipment = "dumbbells"
user_difficulty = "Beginner"
user_weight = 180

classified_profile = classify_user_profile(
    goal=user_goal,
    equipment=user_equipment,
    difficulty=user_difficulty
)

print("DEEP LEARNING / PROFILE CLASSIFIER OUTPUT:")
print(classified_profile)

prompt = f"""
User goal: {user_goal}
Available equipment: {user_equipment}
Difficulty level: {user_difficulty}
Weight: {user_weight} lbs

Classified user profile:
{classified_profile}

Please create a personalized fitness and nutrition plan.
Use the tools if needed.
"""

response = agent.invoke({
    "messages": [
        {"role": "user", "content": prompt}
    ]
})

print("\nFITGUIDE AGENT RESPONSE:")
print(response)

DEEP LEARNING / PROFILE CLASSIFIER OUTPUT:
{'plan_type': 'fat loss', 'equipment_class': 'Dumbbell', 'difficulty': 'Beginner'}

FITGUIDE AGENT RESPONSE:
{'messages': [HumanMessage(content="\nUser goal: I want to lose fat\nAvailable equipment: dumbbells\nDifficulty level: Beginner\nWeight: 180 lbs\n\nClassified user profile:\n{'plan_type': 'fat loss', 'equipment_class': 'Dumbbell', 'difficulty': 'Beginner'}\n\nPlease create a personalized fitness and nutrition plan.\nUse the tools if needed.\n", additional_kwargs={}, response_metadata={}, id='039caf96-f7ed-4a92-be5d-006ab52f1137'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'macro_estimator', 'arguments': '{"weight_lbs": 180, "goal": "fat loss"}'}, '__gemini_function_call_thought_signatures__': {'14b6c1fb-ca3e-478e-b1f2-237654fec628': 'CqIGAQw51sf7VReNmFptukJLvuS5YMq/5jEkXd2PC2Gu2YEH0GECRC5F/lezkAkSM6Y01E4ujBHcmHb2b01TYdCChgzCGBxJUKSvVai9mS8F+XnVY5vbjpv8mP/TlqcoP14cXuTRYWfYYxYqMg9VCEzzQyQVcpALgkkJDB0pHukh4dpguE4q5

# Step 10 — Add Session-Based Memory
This cell creates a simple memory store for progress notes. This allows the agent to remember information such as completed workouts or progress updates during the current notebook session.

In [13]:
# Simple memory store for FitGuide
# This keeps track of user progress notes during the session.

progress_memory = []

def save_progress(user_name, note):
    progress_memory.append({
        "user_name": user_name,
        "note": note
    })
    return f"Progress saved for {user_name}: {note}"

def view_progress(user_name):
    user_notes = [
        item["note"] for item in progress_memory
        if item["user_name"].lower() == user_name.lower()
    ]

    if not user_notes:
        return f"No progress notes found for {user_name}."

    return user_notes

print("Memory functions loaded successfully.")

Memory functions loaded successfully.


# Step 11 — Add Memory Tools
This cell turns the memory functions into LangChain tools. The agent can now save progress notes and retrieve them later when creating updated recommendations.

In [14]:
from langchain.tools import tool

@tool
def save_progress_note(user_name: str, note: str) -> str:
    """
    Saves a user's fitness or nutrition progress note into memory.
    Use this when the user reports progress, completed workouts, weight changes, or consistency updates.
    """
    return save_progress(user_name, note)


@tool
def view_progress_notes(user_name: str) -> str:
    """
    Retrieves saved progress notes for a user.
    Use this when creating updated recommendations based on past progress.
    """
    return str(view_progress(user_name))


# Rebuild tools list with memory tools included
tools = [
    lookup_exercises,
    nutrition_lookup,
    macro_estimator,
    save_progress_note,
    view_progress_notes
]

print("Memory tools added successfully.")
print("Total tools:", len(tools))

Memory tools added successfully.
Total tools: 5


# Step 12 — Recreate FitGuide Agent with Memory Tools
Because new memory tools were added, this cell recreates the agent with all tools included. The updated system prompt tells the agent when to save progress and when to check memory.

In [15]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.4
)

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="""
You are FitGuide, an AI fitness and nutrition coach agent.

You have access to tools for:
- looking up exercises
- looking up nutrition information
- estimating calories and protein
- saving user progress notes
- viewing user progress notes

Use tools whenever they help answer the user's request.

Important behavior:
1. If the user gives a new fitness goal, create a simple beginner-friendly plan.
2. If the user reports progress, save it using the memory tool.
3. If the user asks for an updated plan, check memory first.
4. Keep advice safe and general.
5. Do not claim to be a doctor, dietitian, or certified trainer.

Always respond using this format when creating a plan:

1. Goal Summary
2. Workout Recommendation
3. Nutrition Guidance
4. Estimated Calories and Protein
5. Weekly Coaching Tip
"""
)

print("FitGuide agent recreated successfully with memory tools.")

FitGuide agent recreated successfully with memory tools.


# Step 13 — Scenario 1: Save User Progress
This demo scenario tests the memory feature. The user reports completed workouts and calorie consistency, and the agent should save that progress using the memory tool.

In [16]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "My name is Alex. I completed 2 dumbbell workouts this week and stayed close to my calorie goal."
        }
    ]
})

print(response["messages"][-1].content)

Great job, Alex! Consistency is key, and completing 2 workouts and sticking to your calorie goal is excellent progress. Keep up the fantastic work!


# Step 14 — Verify Memory Was Saved
This cell prints the memory list directly so we can prove that the progress note was saved. This is useful evidence for the final demo because it shows a real state change.

In [17]:
print(progress_memory)

[{'user_name': 'Alex', 'note': 'Completed 2 dumbbell workouts this week and stayed close to calorie goal.'}]


# Step 15 — Scenario 2: Update Plan Using Memory
This cell asks the agent to create an updated plan based on saved progress. The goal is to show that the agent can retrieve memory and personalize future recommendations.

In [18]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "My name is Alex. Based on my progress, give me an updated fat loss plan."
        }
    ]
})

print(response["messages"][-1].content)

[{'type': 'text', 'text': "Here is your updated fat loss plan, Alex, building on your great progress of completing 2 dumbbell workouts and staying close to your calorie goal!\n\n1.  **Goal Summary**\n    Continue with your fat loss journey, building on your consistent efforts.\n\n2.  **Workout Recommendation**\n    *   **Strength Training:** Aim for 3 full-body dumbbell workouts per week. Focus on compound movements. Here are some beginner-friendly dumbbell exercises:\n        *   Goblet Squat: 3 sets of 10-12 reps\n        *   Dumbbell Lunges: 3 sets of 10 reps each leg\n        *   Dumbbell Romanian Deadlift: 3 sets of 10-12 reps\n        *   Dumbbell Step-Up: 3 sets of 10 reps each leg\n    *   **Cardio:** Incorporate 2-3 sessions of moderate-intensity cardio (e.g., brisk walking, jogging, cycling) for 30-45 minutes on non-strength training days.\n    *   **Flexibility:** Include 5-10 minutes of stretching after each workout.\n\n3.  **Nutrition Guidance**\n    *   Continue to priori

# Step 16 — Clean Final Response Display
LangChain responses can include extra metadata. This helper function prints only the final human-readable text so the notebook and demo look cleaner.

In [19]:
def print_final_response(response):
    final_message = response["messages"][-1].content

    if isinstance(final_message, list):
        for item in final_message:
            if isinstance(item, dict) and item.get("type") == "text":
                print(item.get("text"))
    else:
        print(final_message)

print("Clean response printer ready.")

Clean response printer ready.


# Step 17 — Display Clean Response
This cell uses the clean print helper to show only the final answer from the previous agent response.

In [20]:
print_final_response(response)

Here is your updated fat loss plan, Alex, building on your great progress of completing 2 dumbbell workouts and staying close to your calorie goal!

1.  **Goal Summary**
    Continue with your fat loss journey, building on your consistent efforts.

2.  **Workout Recommendation**
    *   **Strength Training:** Aim for 3 full-body dumbbell workouts per week. Focus on compound movements. Here are some beginner-friendly dumbbell exercises:
        *   Goblet Squat: 3 sets of 10-12 reps
        *   Dumbbell Lunges: 3 sets of 10 reps each leg
        *   Dumbbell Romanian Deadlift: 3 sets of 10-12 reps
        *   Dumbbell Step-Up: 3 sets of 10 reps each leg
    *   **Cardio:** Incorporate 2-3 sessions of moderate-intensity cardio (e.g., brisk walking, jogging, cycling) for 30-45 minutes on non-strength training days.
    *   **Flexibility:** Include 5-10 minutes of stretching after each workout.

3.  **Nutrition Guidance**
    *   Continue to prioritize whole, unprocessed foods.
    *   Foc

# Demo Scenario 1 — New Fat Loss User
This is one of the final demo scenarios. The agent receives a beginner fat-loss request, uses the classifier and tools, and returns a complete personalized plan.

In [21]:
user_goal = "I want to lose fat"
user_equipment = "dumbbells"
user_difficulty = "Beginner"
user_weight = 180

classified_profile = classify_user_profile(user_goal, user_equipment, user_difficulty)

prompt = f"""
User goal: {user_goal}
Available equipment: {user_equipment}
Difficulty level: {user_difficulty}
Weight: {user_weight} lbs

Classified user profile:
{classified_profile}

Please create a personalized fitness and nutrition plan.
Use the tools if needed.
"""

response = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
print_final_response(response)

Here is your personalized fitness and nutrition plan to help you achieve your fat loss goal:

### 1. Goal Summary
Your primary goal is fat loss. This plan focuses on a combination of strength training with dumbbells and mindful nutrition to help you reduce body fat while maintaining muscle mass.

### 2. Workout Recommendation
Perform this full-body dumbbell workout 3 times a week on non-consecutive days (e.g., Monday, Wednesday, Friday). Aim for 30-45 minutes per session.

*   **Goblet Squat**: 3 sets of 10-12 reps
*   **Dumbbell Lunges**: 3 sets of 10 reps each leg
*   **Dumbbell Romanian Deadlift**: 3 sets of 10-12 reps
*   **Dumbbell Step-Up**: 3 sets of 10 reps each leg

In addition to strength training, try to incorporate 20-30 minutes of moderate-intensity cardio (like brisk walking, jogging, or cycling) on 2-3 other days of the week.

### 3. Nutrition Guidance
Focus on a balanced diet rich in whole foods.
*   **Prioritize Protein**: Include a lean protein source with every meal 

# Demo Scenario 2 — Save Progress Memory
This final demo scenario shows the agent saving a progress update for Alex. It demonstrates that the memory tool is part of the agent's workflow.

In [22]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "My name is Alex. I completed 2 dumbbell workouts this week and stayed close to my calorie goal."
        }
    ]
})

print_final_response(response)

Great job, Alex! It's fantastic to hear you completed 2 dumbbell workouts and stayed on track with your calorie goals this week. Consistency is key, and you're doing great! Keep up the excellent work!


# Demo Scenario 3 — Updated Plan Based on Memory
This final demo scenario shows the agent using stored progress information to create an updated fat-loss plan. This demonstrates the memory requirement for the final project.

In [23]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "My name is Alex. Based on my progress, give me an updated fat loss plan."
        }
    ]
})

print_final_response(response)

Here is an updated fat loss plan for you, Alex! It looks like you've been consistent with your dumbbell workouts and calorie goals, which is fantastic progress!

1.  **Goal Summary**
    Your goal is fat loss, focusing on consistent effort and healthy habits.

2.  **Workout Recommendation**
    Continue with your 2-3 dumbbell workouts per week. Here are some beginner-friendly dumbbell exercises you can incorporate:
    *   Goblet Squat: 3 sets of 10-12 reps
    *   Dumbbell Lunges: 3 sets of 10 reps each leg
    *   Dumbbell Romanian Deadlift: 3 sets of 10-12 reps
    *   Dumbbell Step-Up: 3 sets of 10 reps each leg
    Aim for 30-45 minutes per session, including a warm-up and cool-down.

3.  **Nutrition Guidance**
    Focus on a balanced diet rich in whole foods. Prioritize lean proteins, plenty of vegetables, fruits, and whole grains. Limit processed foods, sugary drinks, and excessive unhealthy fats. Staying hydrated by drinking plenty of water throughout the day is also crucial fo